# Face Alignment

## Выбор датасета

В этом проекте мы будем работать с датасетом [CelebA - Original Wild Images](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html). Обычно, когда говорят про CelebA имеют ввиду его кропнутую и выровненную версию, но мы будем работать с сырой.

![CelebA](https://figures.semanticscholar.org/7df4f96138a4e23492ea96cf921794fc5287ba72/6-Figure4-1.png)

Оригинальная версия датасета весит очень много: ~20 гб. Все картинки нам не будут нужны, достаточно будет ограничиться 10.000+.

Вашим первым заданием будет подготовить себе рабочий датасет. Он будет использован и в следующих заданиях, поэтому подойдите к этому очень ответственно:

- Скачайте себе датасет CelebA In a Wild любым удобным способом;
- Подумайте над тем, по каким критериям лучше всего выбирать картинки. Используйте файл с атрибутами. Обоснуйте свой выбор текстом. В случае, если обоснования выбора датасета не будет, то баллы могут быть снижены. Отнеситесь к этому серьезно: хорошая подготовка данных очень важна;
- Отберите 10.000+ изображений. Можно и больше при желании;
- При помощи атрибутов bbox'а, обрежьте картинки, чтобы на них остались только лица. При желании можно делать дополнительный кроп, так как не везде разметка идеальна, но не нужно сидеть над каждой картинкой отдельно - потратите слишком много времени;
- Сохраните отдельный csv-файл с оригинальными названиями изображений, которые были отобраны в ваш датасет. В дальнейшем он в том числе понадобится для сдачи проекта.

Несколько практических советов:
- Отнеситесь очень ответственно к отбору данных, это важнейшая часть проекта. Заранее лучше закладывайте время на то, что, очень вероятно, вам придется делать отбор заново по ходу осознания всех заданий.
- Если решили делать дополнительный кроп изображений самостоятельно, то имейте ввиду, что ключевые точки лица в атрибутах указаны в начальной системе координат.
- При первичном анализе и отборе данных не работайте в колабе. Простейшую работу с файлами удобнее делать локально на своем компьютере. Это не требует мощного железа и скачивания библиотек. Если все же пользуетесь колабом, то работайте на CPU, чтобы не лишний раз не тратить токены.
- Если есть возможность, то не удаляйте оригинальный датасет с вашего компьютера на период работы над проектом. Да, это лишняя занятая память, но зато при надобности можно будет быстро что-то изменить.
- Загрузите финальный датасет (10.000+ картинок) себе на Google Disk. Это удобнее, чем каждый раз отдельно загружать его себе в колаб сессию.

In [ ]:
import os, zipfile
# os.system("kaggle datasets download -d jessicali9530/celeba-dataset")

In [ ]:
# with zipfile.ZipFile("celeba-dataset.zip", "r") as zip_ref:
#    zip_ref.extractall("CelebA")

Попробуем получить качественный, разнообразный и хорошо размеченный датасет, который будет содержать лица разных людей, пола, возраста, ракурсов и условий съемки.

Для этого воспользуемся следующими критериями отбора:

1. Использовать только изображения с достаточно крупным лицом - на маленьких лицах теряются детали, ухудшается качество обучения и дальнейшей обработки.

Главный критерий — размер bounding box. Например:
* ширина bbox ≥ 80 px;
* высота bbox ≥ 80 px.

2. Исключить экстремальные ракурсы.

В CelebA нет хорошей разметки yaw/pitch. 
Поэтому лучше использовать Looking Ahead (если есть), либо вообще не фильтровать по позе и оставить разные ракурсы.
Не стоит использовать атрибуты: Big Nose, Big Lips, Pointy Nose,  — они не относятся к ракурсу.
Это сделает датасет более разнообразным.

3. Оставить людей обоих полов.
Использовав аттрибут "Male", сделать примерно 50% Male и 50% Female, чтобы не получить сильный перекос.

4. Оставить людей разных возрастов.

Использовав аттрибут "Young", отфильтровать, например, 70% Young и 30% not Young.
Полностью балансировать не получится, потому что взрослых людей в CelebA намного меньше.


5. Не удалять людей с очками, бородой и т.д.
Наоборот, лучше оставить людей с аттрибутами "Eyeglasses", "Beard", "Mustache", "Hat", - они делают датасет более разнообразным.

6. Удалить совсем плохие изображения. 
Использовать аттрибут "Blurry = -1", если атрибут имеется. Размытые фотографии ухудшают качество.

7. Использовать только уникальные изображения, не брать несколько почти одинаковых кадров подряд одного человека.
CelebA уже достаточно разнообразен, поэтому можно просто случайно выбрать изображения после фильтрации.

#### Обоснование выбора
Такой подход позволяет получить датасет, который:
* содержит качественные лица;
* не имеет большого количества маленьких лиц;
* содержит мужчин и женщин;
* содержит разные возраста;
* содержит разные прически, очки, бороды и аксессуары;
* сохраняет разнообразие поз и освещения.

В дальнейшем такой набор будет лучше подходить для обучения моделей распознавания, генерации и анализа лиц, чем случайная выборка.


In [ ]:
import os
import random

import pandas as pd
from PIL import Image
from tqdm import tqdm

# ==========================================================
# Настройки
# ==========================================================
ROOT = "CelebA"
IMG_DIR = os.path.join(ROOT, "img_celeba")
ATTR_FILE = os.path.join(ROOT, "list_attr_celeba.csv")
BBOX_FILE = os.path.join(ROOT, "list_bbox_celeba.csv")
LANDMARKS_FILE = os.path.join(ROOT, "list_landmarks_align_celeba.csv")

OUTPUT_DIR = os.path.join(ROOT, "cropped_faces")
CSV_OUTPUT = os.path.join(ROOT, "selected_images.csv")

N_IMAGES = 30000

MIN_FACE_WIDTH = 80
MIN_FACE_HEIGHT = 80

PADDING = 0.15
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
os.makedirs(OUTPUT_DIR, exist_ok=True)
# ==========================================================
# Загрузка CSV
# ==========================================================
attrs = pd.read_csv(ATTR_FILE)
bbox = pd.read_csv(BBOX_FILE)

# На всякий случай переименуем первый столбец
attrs.rename(columns={attrs.columns[0]: "image"}, inplace=True)
bbox.rename(columns={bbox.columns[0]: "image"}, inplace=True)

# ==========================================================
# Объединение
# ==========================================================
data = attrs.merge(bbox, on="image")
print(f"Всего изображений: {len(data)}")

# ==========================================================
# Фильтрация по размеру лица
# ==========================================================
data = data[
    (data["width"] >= MIN_FACE_WIDTH) &
    (data["height"] >= MIN_FACE_HEIGHT)
]

print(f"После фильтрации по размеру лица: {len(data)}")

# ==========================================================
# Если есть атрибут Blurry — удаляем размытые фото
# ==========================================================
if "Blurry" in data.columns:
    data = data[data["Blurry"] == -1]
    print(f"После удаления размытых: {len(data)}")

# ==========================================================
# Баланс по полу
# ==========================================================
if "Male" in data.columns:
    male = data[data["Male"] == 1]
    female = data[data["Male"] == -1]
    half = N_IMAGES // 2
    male = male.sample(
        min(len(male), half),
        random_state=RANDOM_STATE
    )
    female = female.sample(
        min(len(female), half),
        random_state=RANDOM_STATE
    )
    selected = pd.concat([male, female])
else:
    selected = data.sample(
        N_IMAGES,
        random_state=RANDOM_STATE
    )

# ==========================================================
# Если после балансировки меньше N_IMAGES
# ==========================================================
if len(selected) < N_IMAGES:
    remain = data.drop(selected.index)
    extra = remain.sample(
        N_IMAGES - len(selected),
        random_state=RANDOM_STATE
    )
    selected = pd.concat([selected, extra])

# Перемешиваем
selected = selected.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)
print(f"Финальный размер датасета: {len(selected)}")

# ==========================================================
# Сохраняем список выбранных изображений
# ==========================================================
selected[["image"]].to_csv(
    CSV_OUTPUT,
    index=False
)
print(f"CSV сохранён: {CSV_OUTPUT}")

# ==========================================================
# Кроп лиц
# ==========================================================
for _, row in tqdm(selected.iterrows(), total=len(selected)):
    image_name = row["image"]
    img_path = os.path.join(IMG_DIR, image_name)
    if not os.path.exists(img_path):
        continue
    img = Image.open(img_path)
    img_w, img_h = img.size

    x = int(row["x_1"])
    y = int(row["y_1"])
    w = int(row["width"])
    h = int(row["height"])

    pad_x = int(w * PADDING)
    pad_y = int(h * PADDING)

    left = max(0, x - pad_x)
    top = max(0, y - pad_y)

    right = min(img_w, x + w + pad_x)
    bottom = min(img_h, y + h + pad_y)

    face = img.crop((left, top, right, bottom))
    face.save(os.path.join(OUTPUT_DIR, image_name))

print("Готово!")

Всего изображений: 202599
После фильтрации по размеру лица: 190183
После удаления размытых: 185249
Финальный размер датасета: 30000
CSV сохранён: CelebA\selected_images.csv


100%|██████████| 30000/30000 [00:01<00:00, 17342.88it/s]

Готово!


## Архитектура Stacked Hourglass Network

В разных вариантах пайплайна для распознавания лиц ключевые точки лица могут предсказываться сразу детектором (MTCNN, RetinaFace и прочие), а могут и отдельной моделью. В этом проекте рассматривается второй вариант. То есть, за детекцию ключевых точек будет отвечать отдельная модель.

**Hourglass** — это U-Net-подобная структура, которая сначала уменьшает разрешение изображения, затем восстанавливает его обратно. Такая структура напоминает по форме песочные часы (hourglass).

**Stacked Hourglass Network** состоит Hourglass-блоков, каждый из которых старается уточнять результат предыдущего. Несмотря на то, что она придумана в 2016 году, до сих пор используется во многих исследовательских проектах для задачи обнаружения ключевых точек.

![image](https://img2018.cnblogs.com/blog/900393/201907/900393-20190722093153502-1808128161.png)

### Hourglass module

Посмотрим подробнее на структуру **отдельного Hourglass-блока**

![retrt](https://curt-park.github.io/images/stacked_hourglass_networks/fig3.png)

Каждый бокс в этой схеме - это Residual block, который отвечает за извлечение признаков на разных уровнях детализации (вспоминаем про ResNet). Причем, каждый такой блок имеет одинаковую размерность на входе и на выходе.

Downsampling и upsampling можно делать разными способами.

*   Для Downsampling: nn.MaxPool2d или nn.Conv2d
*   Для Upsampling: nn.Upsample или nn.ConvTranspose2d

Основная разница: maxpool и upsample - необучаемые слои в отличие от сверток. Это может как быть как минусом, так и плюсом: чем больше параметров - тем медленее идет процесс обучения (при этом не факт, что результаты будут лучше).

То есть, идейно все практически также как было в U-net: полностью симметричная архитектура, сначала идет преобразование в более низкоразмерное пространство, а потом декодирование обратно с пробросами результатов из соотвествующих слоев энкодера. Разница лишь в том, что теперь каждый кирпичик - это Residual block.

А вот реализация ResidualBlock вам в помощь!

Но можете ее править под себя, если очень хочется.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.skip = nn.Identity() if in_channels == out_channels else nn.Conv2d(in_channels, out_channels, kernel_size=1)

        self.conv1 = nn.Conv2d(in_channels, out_channels // 2, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(out_channels // 2)
        self.conv2 = nn.Conv2d(out_channels // 2, out_channels // 2, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels // 2)
        self.conv3 = nn.Conv2d(out_channels // 2, out_channels, kernel_size=1)
        self.bn3 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = self.skip(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))
        return self.relu(x + residual)

При построении архитектуры Hourglass-блоков **не обязательно полностью повторять архитектуру**, которая представлена на картинке из статьи. Вы можете добавлять больше или меньше модулей, некоторые блоки вообще можно не использовать. В целом, это творческая задача и вы вольны делать так, как вам самим хочется. **Главное - чтобы ваша реализация соотвествовала изначальной идее Hourglass, и итоговые результаты были достаточно хорошими.**

Текущая реализация не копирует статью один в один, но полностью соответствует идее архитектуры:

* каждый уровень состоит из ResidualBlock;
* происходит последовательное уменьшение разрешения (encoder);
* затем восстановление (decoder);
* используются skip-connections;
* несколько Hourglass-блоков объединяются последовательно, причем каждый следующий уточняет предсказание предыдущего.

In [ ]:
class Hourglass(nn.Module):
    """
    Recursive Hourglass module.
    """
    def __init__(self, depth, channels):
        super().__init__()
        self.depth = depth
        self.up1 = ResidualBlock(channels, channels)
        self.pool = nn.MaxPool2d(2)
        self.low1 = ResidualBlock(channels, channels)

        if depth > 1:
            self.low2 = Hourglass(depth - 1, channels)
        else:
            self.low2 = ResidualBlock(channels, channels)

        self.low3 = ResidualBlock(channels, channels)
        self.upsample = nn.Upsample(scale_factor=2,
                                    mode='nearest')

    def forward(self, x):
        # skip branch
        up = self.up1(x)

        # encoder
        low = self.pool(x)
        low = self.low1(low)

        # bottleneck
        low = self.low2(low)

        # decoder
        low = self.low3(low)
        low = self.upsample(low)

        return up + low

### Stacked Hourglass Network

Как и было сказано ранее, Stacked Hourglass - это набор одинаковых Hourglass блоков (см. схематический рисунок в начале ноутбука). Но что это за блок между каждыми двумя Hourglass? Чтобы ответить на этот вопрос, нужно сначала разобраться с тем, что мы будем получать на выходе такой нейронной сети.

Предсказывать ключевые точки лица можно поразному. Есть два основных подхода:

1.   Регрессия - предсказывает координаты точек лица напрямую -> $(N, x, y)$.
2.   Heatmap - предсказывает карту вероятностей на выходе, а наиболее подходящие точки находятся через argmax

Не вдваясь в подробности, можно просто сказать, что Heatmap-подход показал себя лучше из-за своей устойчивости к шумам и начальным условиям. В качестве функции потерь в таком случае используют обычный **MSE loss**.

В Stacked Hourglass **используется именно heatmap-подход**. И на выходе каждого Hourglass-блока находится слой (голова), который создает heatmap нужного размера. Обычно это какие-то стандартные варианты по типу *Conv -> BatchNorm -> Relu -> Conv* или просто *Conv*. Каждая heatmap'a прокидывается на следующую голову, и они суммируются, и так, пока слои не закончатся.

Такой подход нужен для реализации **Intermediate Supervision**. Если говорить простыми словами, то это такой вариант обучения нейронной сети, когда мы подсчитываем лосс не только по финальному выходу сети, а также на некоторых промежуточных слоях (головах). Градиенты в этом случае тоже распространяются не только через последний выход, но и через промежуточные уровни. Эти головы не влияют на финальное предсказание напрямую, но помогают модели быстрее и лучше учиться. На практике это означет следующее:

Нужно посчитать лосс (таргет для всех одинаковый) для каждой головы отдельно, а потом просуммировать. Далее Pytorch сам построит за вас весь граф вычислений и правильно запустит везде градиенты. В коде это выглядит так:

```
outputs = model(image)  # outputs — список из N heatmaps от разных голов
losses = [loss_function(output, target) for output in outputs]
total_loss = sum(losses)
total_loss.backward()
optimizer.step()
```

Подведем **итоги по архитектуре**.

Stacked Hourglass состоит из Hourglass-блоков, после каждого такого блока идет голова, которая предсказывает heatmap'у. Каждая heatmap'а суммируется с предыдущей. Градиенты при обучении текут с каждой головы, а не только через последний выход сети.

Подробно про Stacked Hourglass Network можно прочитать в [оригинальной статье](https://arxiv.org/pdf/1603.06937).

In [ ]:
class StackedHourglass(nn.Module):

    def __init__(
        self,
        num_stacks=2,
        num_blocks=4,
        num_features=256,
        num_keypoints=68,
    ):
        super().__init__()
        self.num_stacks = num_stacks
        # Stem
        self.pre = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            ResidualBlock(64, 128),
            nn.MaxPool2d(2),

            ResidualBlock(128, 128),
            ResidualBlock(128, num_features),
        )
        self.hourglasses = nn.ModuleList([
            Hourglass(num_blocks, num_features)
            for _ in range(num_stacks)
        ])
        self.features = nn.ModuleList([
            nn.Sequential(
                ResidualBlock(num_features, num_features),
                nn.Conv2d(num_features,
                          num_features,
                          kernel_size=1),
                nn.BatchNorm2d(num_features),
                nn.ReLU(inplace=True),
            )
            for _ in range(num_stacks)
        ])
        self.heads = nn.ModuleList([
            nn.Conv2d(
                num_features,
                num_keypoints,
                kernel_size=1
            )
            for _ in range(num_stacks)
        ])
        # используются только между стеками
        self.merge_features = nn.ModuleList([
            nn.Conv2d(num_features,
                      num_features,
                      kernel_size=1)
            for _ in range(num_stacks - 1)
        ])
        self.merge_preds = nn.ModuleList([
            nn.Conv2d(num_keypoints,
                      num_features,
                      kernel_size=1)
            for _ in range(num_stacks - 1)
        ])

    def forward(self, x):
        x = self.pre(x)
        outputs = []
        for i in range(self.num_stacks):
            hg = self.hourglasses[i](x)
            feature = self.features[i](hg)
            pred = self.heads[i](feature)
            outputs.append(pred)
            if i < self.num_stacks - 1:
                x = (
                    x
                    + self.merge_features[i](feature)
                    + self.merge_preds[i](pred)
                )
        return outputs

## Подготовка датасета для обучения

На этом этапе у вас уже должен быть готовый датасет на основе CelebA In A Wild.

В разметке CelebA всего 5 точек:

1.   Левый глаз
2.   Правый глаз
3.   Нос
4.   Левый уголок рта
5.   Правый уголок рта


Единственная проблема заключается в том, что разметка - это именно точки, а не heatmap'ы. Но можно их сгенерировать самостоятельно при помощи гауссовского распределения вокруг размеченных точек. Вот вам функции в помощь. Можете их тоже редактивовать под себя, если нужно.

In [ ]:
import numpy as np

def create_heatmap(size, landmark, sigma=2):
    """
    Создаёт один heatmap с гауссовым ядром вокруг точки.

    :param size: (height, width) — размер heatmap'а
    :param landmark:(x, y) — координаты точки
    :param sigma
    :return: heatmap массив
    """
    x, y = landmark
    h, w = size

    # Обрезаем координаты, чтобы не выйти за пределы изображения
    x = min(max(0, int(x)), w - 1)
    y = min(max(0, int(y)), h - 1)

    xx, yy = np.meshgrid(np.arange(w), np.arange(h))
    heatmap = np.exp(-((yy - y)**2 + (xx - x)**2) / (2 * sigma**2))
    return heatmap


def landmarks_to_heatmaps(image_shape, landmarks, sigma=2):
    """
    Преобразует список из N точек в набор из N heatmap'ов.

    :param image_shape: исходный размер изображения (H, W)
    :param landmarks: список из N пар координат [(x1, y1), (x2, y2), ..., (xN, yN),]
    :param sigma:
    :return: массив heatmap'ов вида [N, H, W]
    """
    heatmaps = []

    for (x, y) in landmarks:
        hm = create_heatmap(image_shape, (x, y), sigma=sigma)
        heatmaps.append(hm)

    return np.array(heatmaps)

#### Загрузка таблиц

In [ ]:
selected = pd.read_csv(ATTR_FILE)
bbox = pd.read_csv(BBOX_FILE)
landmarks = pd.read_csv(LANDMARK_FILE)

selected.rename(columns={selected.columns[0]: "image"}, inplace=True)
bbox.rename(columns={bbox.columns[0]: "image"}, inplace=True)
landmarks.rename(columns={landmarks.columns[0]: "image"}, inplace=True)

data = (
    selected
    .merge(bbox, on="image")
    .merge(landmarks, on="image")
)

print(len(data))

#### Генерация heatmap

In [ ]:
for _, row in tqdm(data.iterrows(), total=len(data)):
    image_name = row["image"]
    img = Image.open(os.path.join(OUTPUT_DIR, image_name))
    crop_w, crop_h = img.size

    # -------------------------
    # параметры кропа
    # -------------------------
    x = row["x_1"]
    y = row["y_1"]

    w = row["width"]
    h = row["height"]

    pad_x = int(w * PADDING)
    pad_y = int(h * PADDING)

    left = max(0, x - pad_x)
    top = max(0, y - pad_y)

    # -------------------------
    # координаты landmarks
    # относительно кропа
    # -------------------------
    pts = [
        (
            row["lefteye_x"] - left,
            row["lefteye_y"] - top
        ),
        (
            row["righteye_x"] - left,
            row["righteye_y"] - top
        ),
        (
            row["nose_x"] - left,
            row["nose_y"] - top
        ),
        (
            row["leftmouth_x"] - left,
            row["leftmouth_y"] - top
        ),
        (
            row["rightmouth_x"] - left,
            row["rightmouth_y"] - top
        )
    ]
  
    def heatmap(xx, yy, x, y, sigma):
        return np.exp(
            -((xx - x) ** 2 + (yy - y) ** 2) /
            (2 * sigma ** 2)
        )
    '''
для изображений 64×64 — sigma = 1–2;
для изображений 128×128 — sigma = 2–3;
для изображений 224×224 — sigma = 3–5;
для изображений 256×256 — sigma = 4–6.
    '''   
    SIGMA = 4
    heatmaps = landmarks_to_heatmaps(
        (crop_h, crop_w),
        pts,
        sigma=SIGMA
    )

    HEATMAP_DIR = os.path.join(ROOT, "heatmaps")
    np.save(
        os.path.join(
            HEATMAP_DIR,
            image_name.replace(".jpg", ".npy")
        ),
        heatmaps
    )

#### Проверка результата

In [ ]:
import matplotlib.pyplot as plt

sample = np.load("CelebA/heatmaps/000001.npy")
img = Image.open("CelebA/cropped_faces/000001.jpg")

fig, ax = plt.subplots(2, 3, figsize=(10, 7))

ax[0,0].imshow(img)
ax[0,0].set_title("Image")

titles = [
    "Left eye",
    "Right eye",
    "Nose",
    "Left mouth",
    "Right mouth"
]

for i in range(5):
    r = (i + 1) // 3
    c = (i + 1) % 3

    ax[r, c].imshow(sample[i], cmap="jet")
    ax[r, c].set_title(titles[i])

plt.tight_layout()
plt.show()

#### Дополнительная проверка: наложение heatmap на изображение

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.imshow(sample[2], cmap="jet", alpha=0.5)  # heatmap носа
plt.axis("off")
plt.show()

## Обучениe

## Выравнивание по предсказанным точкам

Существует множество вариантов, как по полученным точкам правильно преобразовать картинку. Главное, что вам нужно понимать - **это задача классического компьютерного зрения** и решается при помощи математики, без нейронок. Вдаваться в подробности конкретных алгоритмов мы не будем.

Можно использовать аффинное преобразование, тогда потребуется только 3 точки, можно, например, искать матрицу гомографии, где может быть использовано больше точек, а может быть, есть еще что-то. Реализовывать эти алгоритмы самим не нужно. Достаточно провести небольшой ресерч и найти готовое решение (но **не готовую нейронку для выравнивания**), лишь бы оно работало. Количество используемых точек тоже выбирайте сами, подойдет любой вариант. Условный ориентир для поиска - библиотека **opencv**. Обязательно приведите примеры того, как работает ваш алгоритм.

# План заданий

По итогу, в этом блоке у вас следующие задачи:

*   Подготовить датасет, сохранить файл с оригинальными названиями изображений
*   Реализовать Hourglass блок
*   Реализовать Stacked Hourglass
*   Преобразовать точки лица в Heatmap'ы
*   Обучить Stacked Hourglass
*   Найти функцию, которая по предсказанным ключевым точкам выравнивает лица на картинке (face alignment)
*   Подготовить датасет с кропнутыми и выровненными лицами для следующего этапа

**P.S. Не забывайте сохранять модели после обучения и выводите промежуточные результаты на экран**



**Удачи! У вас всё получится 💗!**